In [ ]:
import pandas as pd 
pd.set_option('display.max_columns', 500)
from tqdm import tqdm
import numpy as np
from matplotlib import pyplot as plt 
#import lightgbm as lgb
from scipy.stats import mode
from sklearn.preprocessing import LabelEncoder
from datetime import timedelta
from pandas import pivot_table

import seaborn as sns
sns.set()
%config InlineBackend.figure_format = 'svg'

### المؤلف سيرجي بولاييف، اسم Slack: @ser-serege، خريف 2018



## الجزء الأول. وصف مجموعة البيانات والميزات



###### تحتوي مجموعة البيانات هذه على سجل معاملات العملاء لمدة 3 أشهر من الاستخدام التفضيلي للمنتج المصرفي.
في ملف الاختبار. يحتوي ملف CSV على سطور من المعاملات التي تمت بواسطة عملاء البنك والتي يبلغ عددها 518375. يحتوي عمود cl_id على معرف العميل الداخلي. بالنسبة لكل cl_id فريد، يجب عليك توقع ما إذا كان العميل سيستمر في استخدام المنتج (target_flag). تشير القيمة 0 إلى الفشل وتشير القيمة 1 إلى استمرار الاستخدام.



| العمود | النسخ |
|---------------|--------------------------------------|
|الفترة |شهر المعاملة |       
|cl_id |معرف العميل |
|MCC |رمز فئة البائع |
|channel_type |قناة جذب العملاء |
|العملة |العملة |
|TRDATETIME |تاريخ/وقت المعاملة |
|المبلغ |مبلغ المعاملة |
|trx_category |نوع المعاملة الدفع عبر نقطة البيع
|               |محطة نقاط البيع، C2C_OUT – التحويل 
|               |(الدفعة الصادرة)، C2C_IN – البطاقة |
|               |المعاملة (الدفعة الواردة)، الإيداع| 
|               |بطاقة في ماكينة الصراف الآلي، WD_ATM_PARTNER – نقدًا |
|               | عمليات السحب لدى شركاء أجهزة الصراف الآلي
|target_flag |هل سيستمر العميل في استخدام المنتج بعد فترة السماح (1/0) (الهدف)
|target_sum | مبلغ نوع المعاملة لنقطة البيع للأشهر الثلاثة المقبلة (الهدف)


In [ ]:
#raw_df = pd.read_csv('Rosbankk.csv',error_bad_lines=False)
#
#raw_df.to_csv('rosbank_train.csv')
#test = pd.read_csv('rosbank_test.csv',error_bad_lines=False)
raw_df = pd.read_csv('Rosbankk.csv',error_bad_lines=False)

In [ ]:
#raw_df = pd.read_csv('rosbank_train.csv',error_bad_lines=False)
#del raw_df['Unnamed: 0']

In [ ]:
raw_df.head()

In [ ]:
raw_df['cl_id'].nunique()

In [ ]:
plt.hist(raw_df[raw_df['target_flag'] == 1]['target_flag'].dropna(), color='red', alpha=0.3, bins=30);
plt.hist(raw_df[raw_df['target_flag'] == 0]['target_flag'].dropna(), color='green', alpha=0.5, bins=30);
print(round((raw_df[raw_df['target_flag'] == 1]['cl_id'].nunique() / raw_df['cl_id'].nunique())*100,1), '% of taget = 1')

In [ ]:
raw_df.info()
raw_df.describe()

In [ ]:
print( 'Number of unique clients =',raw_df['cl_id'].nunique())
print ('At channel_type column there are ', round(100*(len(raw_df[raw_df['channel_type'].isna()]) / len(raw_df)),1), '% of empty cells')


##### دعونا نرسم مكان توزيع أبعاد الهدف


In [ ]:
X = raw_df[['cl_id','target_flag']].groupby('cl_id').agg('max').reset_index()
ind = X['target_flag']==0
plt.plot(X['cl_id'][ind], np.random.rand(np.sum(ind)), 'g.', label='negative case')
ind = X['target_flag']==1
plt.plot(X['cl_id'][ind], np.random.rand(np.sum(ind)), 'b.', label='positive case')
plt.legend()

#من مجموعة البيانات نرى أن:
مركز عملائي (MCC) ليس رقمًا صحيحًا، بل يجب أن يكون بيانات فئة (سنجد أوصافًا في الإنترنت)؛  يجب أن تكون العملة أيضًا بيانات فئة
في المجموع هناك 490513 معاملة لمدة 3 أشهر، من خلال 5000 عميل. بشكل عام يتعلق الأمر 
98 معاملة لعميل واحد لمدة 3 أشهر من استخدام البطاقة.
الفترة الزمنية من 01/01/2017 إلى 01/12/2017



من الوصف نرى أن هناك خلايا فارغة في ميزة Channel_type. دعونا نملأها في "type6"


In [ ]:
raw_df.channel_type.unique()

In [ ]:
#From description we see that there are empty cells in channel_type feature. Let's fill them into 'type6'

raw_df.channel_type.fillna('type6', inplace = True)


من الوصف نرى أن PERIOD و TRDATETIME لهما نوع الكائن. وشكل غريب . دعونا نحللها ونحولها إلى تنسيق التاريخ والوقت


In [ ]:
from datetime import datetime, date, time

raw_df['PERIOD'] = raw_df['PERIOD'].apply(pd.to_datetime)

# Creating separate cols for yr, month,...
raw_df['Year'] = raw_df.TRDATETIME.str[5:7]
raw_df['Month'] = raw_df.TRDATETIME.str[2:5]
raw_df['Date'] = raw_df.TRDATETIME.str[0:2]
raw_df['Hour'] = raw_df.TRDATETIME.str[8:10]

# Replace month with ints
raw_df.Month = raw_df.Month.replace(to_replace=['JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN','JUL','AUG',\
                                                'SEP','OCT','NOV','DEC' ], value=[1,2,3,4,5,6,7,8,9,10,11,12])

raw_df.Year = raw_df.Year.apply(pd.to_numeric)
raw_df.Date = raw_df.Date.apply(pd.to_numeric)
raw_df.Month= raw_df.Month.apply(pd.to_numeric)
raw_df.Hour = raw_df.Hour.apply(pd.to_numeric)
raw_df.Year = raw_df.Year + 2000

# making date format
def to_date(row):    
    return date(row[10], row[11], row[12])
raw_df['DateFormat'] = raw_df.apply(to_date, axis=1)

# making Quater of the Year feature 
def Quater(row):
    if row['Month']in [1, 2, 3]:
        return 1
    if row['Month']in [4, 5, 6]:
        return 2   
    if row['Month']in [7, 8, 9]:
        return 3 
    if row['Month']in [10, 11, 12]:
        return 4

# Applying features is the day is weekend and quater
raw_df['quater_of_year'] = raw_df.apply(Quater, axis = 1)
raw_df['weekend'] = raw_df['DateFormat'].astype('datetime64[ns]')
raw_df['weekend'] = ((raw_df.weekend.dt.dayofweek) // 5 ==1).astype(float)

In [ ]:
raw_df.currency.unique()

In [ ]:
raw_df.trx_category.unique()


##### توجد في لغة بايثون مكتبة يمكنها تحويل العملات. دعونا نحول جميع المبالغ إلى روبل


In [ ]:
from currency_converter import CurrencyConverter

converter = CurrencyConverter(fallback_on_missing_rate=True, fallback_on_wrong_date=True)
converter_currencies = converter.currencies

def convert_to_rub(amount, currency, day):
    if currency == 'RUB':
        return amount
    else:
        if currency in converter_currencies:
            return converter.convert(amount, currency, 'RUB', date = day)
        else: amount     
        return amount
    
# also from task descripttion we know that there are cash in and cash out . 
# It that logic make the functions which convert in (+) or (-)
    
def cash_in_out(raw):
    if raw['trx_category'] == 'POS':
        return raw['amount']*(-1)
    if raw['trx_category'] == 'C2C_OUT':
        return raw['amount']*(-1)
    if raw['trx_category'] == 'WD_ATM_PARTNER':
        return raw['amount']*(-1)
    if raw['trx_category'] == 'WD_ATM_ROS':
        return raw['amount']*(-1)    
    else:
        return raw['amount']
    
raw_df['amount'] = raw_df.apply(lambda x: convert_to_rub(x['amount'], x['currency'], x['DateFormat']), axis = 1)
raw_df['cash_in_out'] = raw_df.apply(cash_in_out, axis=1)


#### لنقم بإنشاء ميزة تصف العملات روب، دولار، يورو أخرى


In [ ]:
def Is_rub(raw):
    if raw['currency'] == 810:
        return 'Rub'
    if raw['currency'] == 643:
        return 'Rub'
    if raw['currency'] ==840:
        return '$'
    if raw['currency'] == 978:
        return 'Euro'
    else: 
        return 'other'
raw_df['cur']= raw_df.apply(Is_rub, axis=1)


###### السماح بإنشاء ميزة بداية ونهاية فترة استخدام البطاقة


In [ ]:
max_date = raw_df[['cl_id', 'DateFormat']].groupby('cl_id').max().reset_index()
max_date.columns = ['cl_id', 'last_action']

raw_df = pd.merge(raw_df, max_date, how='left', on='cl_id')


##### لدي فرضية مفادها أنه إذا استمر العميل في استخدام البطاقة بشكل فعال في نهاية فترة الاستخدام التفضيلي، فسوف يستمر في استخدام البطاقة بعد ذلك. لذلك دعونا ننشئ ميزات لآخر 14 يومًا وآخر 30 يومًا قبل نهاية الفترة


In [ ]:
raw_df['last_action'] = pd.to_datetime(raw_df['last_action'])

raw_df['last_14_days'] = raw_df['last_action'] - timedelta(days=14)
raw_df['last_30_days'] = raw_df['last_action'] - timedelta(days=30)

raw_df['DateFormat'] = pd.to_datetime(raw_df['DateFormat'])
raw_df['last_14_days']= pd.to_datetime(raw_df['last_14_days'])
raw_df['last_30_days'] = pd.to_datetime(raw_df['last_30_days'])

def last_14_days1(row):
    if row['DateFormat']>=row['last_14_days']:
        return 1

def last_30_days1(row):
    if row['DateFormat']>=row['last_30_days']:
        return 1
    
raw_df['last_14_days'] = raw_df.apply(last_14_days1, axis=1)
raw_df['last_30_days'] = raw_df.apply(last_30_days1, axis=1)


##### السؤال التالي الذي يجب حله هو رموز MCC (رمز فئة التاجر). هذه هي الرموز التي تحدد نوع العملية التي يقوم بها العميل. ل 


In [ ]:
mcc_codes = pd.read_excel('mcc_codes1.xlsx')
mcc_codes.columns = ['MCC', 'Name' , 'Group']

raw_df = pd.merge(raw_df, mcc_codes, 'left', on=['MCC'])

In [ ]:
mcc_codes.head()


##### رموز مركز عملائي لها اسم حالي واسم مجمع. سوف نستخدمها لوظائف التجميع
##### أحد رموز مركز عملائي يعني الاسترداد النقدي من عمليات نقاط البيع


In [ ]:
def cashback(raw):
    if raw['trx_category'] == 'POS':
        return raw['amount']*0.02
raw_df['cashback'] = raw_df.apply(cashback, axis=1)


#### أخيرا وصلنا


In [ ]:
raw_df.head()


##### جميع NaN تعني أنها تساوي 0


In [ ]:
raw_df = raw_df.fillna(0)

In [ ]:
# Save it fo file 
#raw_df.to_csv('rosbank_train1.csv')
#raw_df = pd.read_csv('rosbank_train1.csv')


### حسنًا، يبدو أننا قمنا بإعداد مجموعة بيانات لمزيد من وظائف التجميع. 


In [ ]:
raw_df.info()

In [ ]:
raw_df.head(2)

##### القيام ببعض التجميعات لإنشاء عينة من العملاء الفريدين. النهج العام هو حساب الميزات الفئوية والتجميع حسب الأرقام ['max'، 'min'، 'mean'، 'count'، 'sum']


In [ ]:
def days_in_use(x):
    return (np.max(x) - np.min(x)).days

days_usage = raw_df[['cl_id','DateFormat']].groupby('cl_id').agg(days_in_use)

days_usage['target_flag'] = raw_df['target_flag']

max_date = raw_df[['cl_id', 'DateFormat']].groupby('cl_id').max()
days_usage['days_from_end_period']= (max(raw_df['DateFormat']) - max_date['DateFormat']).dt.days


num_trans_total = raw_df[['cl_id','DateFormat']].groupby('cl_id').agg('count').reset_index()
num_trans_total.columns = ['cl_id', 'num_trans_total']
num_trans_total.index=num_trans_total.cl_id
days_usage['Num_trans_total'] = num_trans_total.num_trans_total

num_trans_month = raw_df[['cl_id','Month']].groupby('cl_id').agg(['max', 'min', 'mean', 'count', 'sum']).reset_index()
#num_trans_month.columns = ['cl_id', 'num_trans_month']
num_trans_month.index=num_trans_month.cl_id
days_usage[num_trans_month.columns] = num_trans_month


balance_on_end_of_period = raw_df[['cl_id', 'cash_in_out']].groupby('cl_id').agg(['max', 'min', 'mean', 'count','sum']).reset_index()
#balance_on_end_of_period.columns=['cl_id', 'balance_on_end_of_period']
balance_on_end_of_period.index=balance_on_end_of_period.cl_id
days_usage[balance_on_end_of_period.columns] = balance_on_end_of_period

cashback = raw_df[['cl_id', 'cashback']].groupby('cl_id').sum().reset_index()
cashback.columns=['cl_id', 'cashback']
cashback.index=cashback.cl_id
days_usage['cashback'] = cashback.cashback


spent_trx_category = raw_df[['cl_id', 'trx_category' ,'amount']].groupby(['cl_id', 'trx_category']).sum().\
                                                                                        unstack().reset_index()
spent_trx_category=spent_trx_category.fillna(0)
spent_trx_category.columns = ['cl_id', 'BACK_TRX', 'C2C_IN', 'C2C_OUT', 'CASH_ADV', 'CAT', 'DEPOSIT',
       'POS', 'WD_ATM_OTHER', 'WD_ATM_PARTNER', 'WD_ATM_ROS'] 
spent_trx_category.index= spent_trx_category.cl_id

days_usage[['BACK_TRX', 'C2C_IN', 'C2C_OUT', 'CASH_ADV', 'CAT', 'DEPOSIT',
       'POS', 'WD_ATM_OTHER', 'WD_ATM_PARTNER', 'WD_ATM_ROS']] = spent_trx_category[['BACK_TRX', 'C2C_IN', 'C2C_OUT',\
                                 'CASH_ADV', 'CAT', 'DEPOSIT','POS', 'WD_ATM_OTHER', 'WD_ATM_PARTNER', 'WD_ATM_ROS']]

quntity_of_mcc = raw_df[['cl_id','MCC']].groupby(['cl_id','MCC']).apply(lambda x: x.count()).unstack().\
                                                                                        max(axis=1).reset_index()
quntity_of_mcc.columns=['cl_id', 'quntity_of_mcc']
quntity_of_mcc.index= quntity_of_mcc.cl_id
days_usage['quntity_of_mcc']=quntity_of_mcc.quntity_of_mcc




multy_currency = raw_df[['cl_id', 'currency']].groupby(['cl_id', 'currency']).first().reset_index()
multy_currency = multy_currency.groupby(['cl_id']).count()
#multy_currency.index= multy_currency.cl_id

days_usage['multy_currency']=multy_currency.currency



last_14_days=raw_df[['cl_id', 'last_14_days']].groupby(['cl_id']).agg('sum')
last_30_days=raw_df[['cl_id', 'last_30_days']].groupby(['cl_id']).agg('sum')

days_usage[last_14_days.columns]=last_14_days
days_usage[last_30_days.columns]=last_30_days




group_mcc = pivot_table(raw_df, values='cash_in_out', 
                    index=['cl_id'], columns=['Group'], aggfunc=lambda cash_in_out: len(cash_in_out.unique())).fillna(0)
group_mcc2 = pivot_table(raw_df, values='cash_in_out', 
                    index=['cl_id'], columns=['Group'], aggfunc=np.sum).fillna(0)

mcc = pd.merge(group_mcc, group_mcc2, 'left', on=days_usage.index)

mcc.index=mcc.key_0
days_usage[mcc.columns]= mcc
del days_usage['key_0']

trx_category = raw_df[['cl_id','trx_category', 'cash_in_out']].groupby(['cl_id','trx_category']).agg(['max', \
                                                                'min', 'mean', 'count', 'sum']).unstack()
days_usage[trx_category.columns]=trx_category


quater_of_year = raw_df[['cl_id', 'quater_of_year']].groupby('cl_id').agg('sum').reset_index()
#quater_of_year.columns=['cl_id', 'quater_of_year']
quater_of_year.index=quater_of_year.cl_id
days_usage[quater_of_year.columns] = quater_of_year



last_action = raw_df[['cl_id', 'last_action']].groupby('cl_id').count().reset_index()
last_action.columns=['cl_id', 'last_action']
last_action.index=last_action.cl_id
days_usage['last_action'] = last_action.last_action

cur = raw_df[['cl_id','cur', 'cash_in_out']].groupby(['cl_id','cur']).agg(['max', 'min', 'mean', 'count', 'sum'\
                                                                          ]).unstack()
#cur.index=cur.cl_id
days_usage[cur.columns]=cur


last_14_days = Raw_df[['cl_id', 'last_14_days']].groupby('cl_id').agg(['max', 'min', 'mean', 'count', 'sum'\
                                                                      ]).reset_index()
#last_14_days.columns=['cl_id', 'last_14_days']
last_14_days.index=last_14_days.cl_id
days_usage[last_14_days.columns] = last_14_days
last_30_days = Raw_df[['cl_id', 'last_30_days']].groupby('cl_id').agg(['max', 'min', 'mean', 'count', 'sum'\
                                                                      ]).reset_index()
#last_30_days.columns=['cl_id', 'last_30_days']
last_30_days.index=last_14_days.cl_id
days_usage[last_30_days.columns] = last_30_days



##### ميزة أخرى يمكن أن تكون العلاقة بين (جميع عدد المعاملات) / إلى (عدد المعاملات خلال 14 يومًا آخر و 30 يومًا آخر)


In [ ]:
days_usage['all_to_last14'] =  days_usage['last_14_days'] / days_usage['Num_trans_total']
days_usage['all_to_last30'] =  days_usage['last_30_days'] / days_usage['Num_trans_total']


#### لدينا مجموعة بيانات تحتوي على 139 ميزة مجمعة


In [ ]:
days_usage.head()

In [ ]:
days_usage.describe()


##### من المؤكد أن هناك الكثير من NaN بسبب التجميعات. لذا، إذا كان Nan، فاملأه بمقدار 0


In [ ]:
days_usage=days_usage.fillna(0)

In [ ]:
days_usage.

In [ ]:
days_usage.head()

In [ ]:
days_usage.to_csv('days_us.csv')

In [ ]:
Plotting visual info. 

In [ ]:
sns.heatmap(days_usage.corr())


#### من الرسم البياني نرى أن هناك الكثير من الميزات المرتبطة. دعونا نجدهم ونسقطهم


In [ ]:
# чтобы убрать все кореллирующие признакие
def drop_corr_col(df_corr):
    upper = df_corr.where(np.triu(np.ones(df_corr.shape),
                          k=1).astype(np.bool))
    to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
    return(to_drop)

In [ ]:
corr=days_usage.corr().abs()
drop_col=drop_corr_col(corr)
print('We found and drop',len(drop_col), 'correlated features with the coefficient more then 0.9')


In [ ]:
#Let's make PCA with 2 components from all of them and then add this two components into dataset , others drop
from sklearn.decomposition import PCA
pca = PCA(n_components=2).fit_transform(days_usage[drop_col])

pca_df = pd.DataFrame(pca, columns=['pca1', 'pca2'])
pca_df.index=days_usage.index

days=days_usage.drop(drop_col, axis=1)
days[pca_df.columns]= pca_df

In [ ]:
days.head()

In [ ]:
days.to_csv('days.csv')

In [ ]:
plt.hist(days[days['target_flag'] == 1]['target_flag'].dropna(), color='red', alpha=0.3, bins=30);
plt.hist(days[days['target_flag'] == 0]['target_flag'].dropna(), color='green', alpha=0.5, bins=30);

In [ ]:
plt.plot(days['last_14_days'])

In [ ]:
sns.boxplot(x='DateFormat', data=days);

In [ ]:
sns.boxplot(x='Num_trans_total', data=days);

In [ ]:
sns.pairplot(days_usage[['DateFormat','days_from_end_period',
                         'multy_currency', 'quntity_of_mcc'  ] ])


### حسنًا، لقد قمنا بإعداد مجموعة البيانات، وشاهدنا توزيع بعض الميزات. دعونا نختار المقاييس لتقييم جودة النماذج المستقبلية.
##### لدينا مهمة التصنيف الثنائي، لذلك سنرى مقاييس ROCAUC.  لدينا أيضًا خلل في التوازن بين الفئات المستهدفة، لكن الاختلاف ليس كبيرًا جدًا. لذا فإن الدقة والدقة جيدة أيضًا. 


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix, auc
from sklearn.model_selection import KFold, StratifiedKFold

def calc_auc(y_test2, y_pred, plot_label='', prin=True):
    fpr, tpr, _ = roc_curve(y_test2, y_pred)
    auc_val = auc(fpr, tpr)
    if prin:
        print('ROC AUC: {0:.4f}'.format(auc_val))
    if plot_label:
        plt.plot(fpr, tpr, label=plot_label)
        plt.xlabel('FPR')
        plt.ylabel('TPR')
    return auc_val


### قم أيضًا بإنشاء وظيفة لرسم مصفوفة الارتباك


In [ ]:
def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)

    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, cm[i, j],
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

font = {'size' : 15}

plt.rc('font', **font)


#### نحن على استعداد لبناء النماذج.للقيام بذلك سوف نقوم بإنشاء Train_test_split. نظرًا لأن لدينا فصولًا غير متوازنة وعدد أقل من المستخدمين الأقدم، فسوف نقوم بهزهم من خلال تقسيم عشوائي طبقي. 


In [ ]:
X = days.copy()
y = days.target_flag
X = X.reset_index()
X.drop([('cl_id', '')], axis=1)
del X['target_flag']
del X[('cl_id', '')]

In [ ]:
X.head()

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

splitter = StratifiedShuffleSplit(n_splits=2, test_size=0.3, random_state=17)

for train_index, test_index in splitter.split(X, y):
    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]
    
    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]

In [ ]:
import xgboost
from sklearn.metrics import roc_auc_score, roc_curve
xgb = xgboost.XGBClassifier(learning_rate=0.1, max_depth=5, n_jobs=-1)
xgb.fit(X_train, y_train)
y_train_predict = xgb.predict_proba(X_train)[:, 1]
y_test_predict = xgb.predict_proba(X_test)[:, 1]
roc_auc_train = np.round(roc_auc_score(y_train, y_train_predict), 2)
roc_auc_test = np.round(roc_auc_score(y_test, y_test_predict), 2)
print("Train: ", roc_auc_train)
print("Test: ", roc_auc_test)

In [ ]:
from xgboost import plot_importance
plot_importance(xgb, max_num_features = 15)


## نرى أن cl_id عبارة عن تسرب للبيانات. أسقطه.


In [ ]:
del X_train['cl_id']
del X_test['cl_id']

In [ ]:
import xgboost
from sklearn.metrics import roc_auc_score, roc_curve
xgb = xgboost.XGBClassifier( n_jobs=-1)
xgb.fit(X_train, y_train)
y_train_predict = xgb.predict_proba(X_train)[:, 1]
y_test_predict = xgb.predict_proba(X_test)[:, 1]
roc_auc_train = np.round(roc_auc_score(y_train, y_train_predict), 2)
roc_auc_test = np.round(roc_auc_score(y_test, y_test_predict), 2)
print("Train: ", roc_auc_train)
print("Test: ", roc_auc_test)

In [ ]:
from xgboost import plot_importance
plot_importance(xgb, max_num_features = 15)


نتائج سيئة. دعونا نبني الانحدار اللوجستي البسيط


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression, LogisticRegression

scaler= StandardScaler()
X_scaled_train = scaler.fit_transform(X_train)
X_scaled_test = scaler.transform(X_test)

lr=LogisticRegression()
lr.fit(X_scaled_train,y_train)
y_pred_train= lr.predict_proba(X_scaled_train)
y_pred_test= lr.predict_proba(X_scaled_test)

#.fit(x_train1_l1,y_train).score(x_train1_l1,y_train)
#print(score)
print(np.round(roc_auc_score(y_train, y_pred_train[:,1]), 2))
print(np.round(roc_auc_score(y_test, y_pred_test[:,1]), 2))

y_pred_rf_test1 = lr.predict_proba(X_scaled_test)[:, 1]
y_pred_rf_train1 = lr.predict_proba(X_scaled_train)[:, 1]

print('Train:')
calc_auc(y_train, y_pred_rf_train1, 'train')
print('Test:')
calc_auc(y_test, y_pred_rf_test1, 'test')
plt.legend();


## أفضل قليلاً من العشوائي


In [ ]:
from catboost import CatBoost, CatBoostClassifier

model = CatBoostClassifier( )

model.fit(
    X_train, y_train,
    #cat_features=categorical_features_indices,
    eval_set=(X_test, y_test),
    logging_level='Silent',
    plot=True
);


## تركيب مبالغ فيه وأفضل نتيجة حتى الآن


In [ ]:
y_pred_rf_test1 = model.predict_proba(X_test)[:, 1]
y_pred_rf_train1 = model.predict_proba(X_train)[:, 1]

print('Train:')
calc_auc(y_train, y_pred_rf_train1, 'train')
print('Test:')
calc_auc(y_test, y_pred_rf_test1, 'test')
plt.legend();


لنجرب GridserchCV مع XGGClassifier. 
مع التحقق من الصحة عبر 
عدد الطيات = 5 
التهديف = روك الجامعة الأمريكية
خلط صحيح
ومعلمات مختلفة 


In [ ]:
from sklearn.cross_validation import *
from sklearn.grid_search import GridSearchCV
import xgboost as xgb

parameters = {'nthread':[4], #when use hyperthread, xgboost may become slower
              'objective':['binary:logistic'],
              'learning_rate': [0.01,0.1,1 ], #so called `eta` value
              'max_depth': [4,10],
              'silent': [1],
              'subsample': [0.8],
              'colsample_bytree': [0.7],
              'n_estimators': [500, 1000], 
              'missing':[-999],
              'seed': [1337]}

xgb_model = xgb.XGBClassifier()
clf = GridSearchCV(xgb_model, parameters, n_jobs=5, 
                   cv=StratifiedKFold(y_train, n_folds=5, shuffle=True), 
                   scoring='roc_auc',
                   verbose=2, refit=True)

In [ ]:
del X_train['target_flag']

In [ ]:
clf.fit(X_train, y_train)

In [ ]:
y_pred_clf_test1 = clf.predict_proba(X_test)[:, 1]
y_pred_clf_train1 = clf.predict_proba(X_train)[:, 1]

print('Train:')
calc_auc(y_train, y_pred_clf_train1, 'train')
print('Test:')
calc_auc(y_test, y_pred_clf_test1, 'test')
plt.legend();

In [ ]:
from sklearn.metrics import precision_recall_curve, classification_report
report = classification_report(y_test, clf.predict(X_test))
print(report)


### لذلك، نتوقع أن نصف العينات تقريبًا غير صحيحة.


In [ ]:
import itertools
cnf_matrix = confusion_matrix(y_test, clf.predict(X_test))
plt.figure(figsize=(10, 6))
plot_confusion_matrix(cnf_matrix, classes=['Not_continue', 'Continue to use'],
                      title='Confusion matrix')
plt.savefig("conf_matrix.png")
plt.show()


## الخلاصة
أفضل نموذج كان Catboost
لقد قمنا بالكثير من عمليات هندسة الميزات والمعالجة المسبقة للبيانات، ولكن لا يمكننا التنبؤ بهذا العميل الذي لن يستخدم بطاقتنا.
لمزيد من علينا إنشاء المزيد من الميزات: 
1. لدمج الأرقام في مجموعات
2. Cout WOE (وزن الأدلة على هذه المجموعات
3. عمل سلسلة زمنية من المعاملات لكل عميل
4. كما أن 5000 من العملاء ليسوا عينة ممثلة